<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/day1/notebooks/4_bow_limits_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 1 — Seeing the Limits of Bag-of-Words  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every **✏️ Exercise** is filled in with one possible
answer, plus a short comment on what it shows.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Setup complete!")

In [ ]:
def make_dtm(docs, **kwargs):
    """Return a DTM as a readable DataFrame (rows = docs, columns = terms)."""
    vec = CountVectorizer(**kwargs)
    X = vec.fit_transform(docs)
    return pd.DataFrame(
        X.toarray(),
        columns=vec.get_feature_names_out(),
        index=[f"D{i+1}" for i in range(len(docs))],
    )

make_dtm(["the cat sat", "the dog sat"])

## 1. Word order is lost

In [ ]:
pair = [
    "the dog bit the man",
    "the man bit the dog",
]
dtm = make_dtm(pair)
dtm

In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(pair)
sim = cosine_similarity(X)[0, 1]
print(f"Cosine similarity between the two sentences: {sim:.3f}")
print("(1.000 means the model considers them identical.)")

> **✏️ Exercise 1**
>
> Come up with your own pair of sentences that mean different things but use the same words.
> Check their cosine similarity. Can you get anything other than 1.0?


In [ ]:
# ✅ Solution
my_pair = [
    "money can buy happiness",
    "happiness can buy money",
]
X = CountVectorizer().fit_transform(my_pair)
print("Cosine similarity:", round(cosine_similarity(X)[0, 1], 3))

# Comment: it is 1.0 again. As long as the two sentences contain the same words the
# same number of times, their bag-of-words vectors are identical — no matter how you
# reorder them. You literally cannot get below 1.0 by reordering alone. That is the
# whole point: order carries meaning the representation throws away.

## 2. No sense of meaning (synonyms look unrelated)

In [ ]:
sentences = [
    "I am very happy",
    "I am very joyful",
    "I am very sad",
]
make_dtm(sentences)

In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(sentences)
sims = cosine_similarity(X)
sim_df = pd.DataFrame(sims, index=["D1 happy", "D2 joyful", "D3 sad"],
                      columns=["D1 happy", "D2 joyful", "D3 sad"])
print("Cosine similarity between sentences:")
sim_df.round(3)

In [ ]:
print(f"happy  vs joyful: {sims[0,1]:.3f}")
print(f"happy  vs sad:    {sims[0,2]:.3f}")
print("\nSame number — the model cannot tell synonyms from opposites.")

> **✏️ Exercise 2**
>
> Add a fourth sentence `"I am very cheerful"`. Recompute the similarity matrix. Is
> "cheerful" any closer to "happy" than "sad" is? Explain why not.


In [ ]:
# ✅ Solution
sentences4 = [
    "I am very happy",
    "I am very joyful",
    "I am very sad",
    "I am very cheerful",
]
vec = CountVectorizer()
X = vec.fit_transform(sentences4)
sims = cosine_similarity(X)

labels = ["happy", "joyful", "sad", "cheerful"]
sim_df = pd.DataFrame(sims.round(3), index=labels, columns=labels)
print(sim_df)
print()
print(f"happy vs cheerful: {sims[0,3]:.3f}")
print(f"happy vs sad:      {sims[0,2]:.3f}")

# Comment: "cheerful" is NOT any closer to "happy" than "sad" is — all the
# non-shared words sit in separate columns, so every pair shares exactly the same
# three words ("i am very") and differs by one unique word. The model has no notion
# that cheerful/joyful/happy belong together and sad does not.

## 3. High dimensionality and sparsity

In [ ]:
corpus = [
    "the economy grew last quarter",
    "climate policy dominated the debate",
    "the football match ended in a draw",
    "researchers published a new study",
    "the election results surprised everyone",
    "inflation affected household budgets",
    "the orchestra performed a symphony",
    "students protested tuition fees",
]
dtm = make_dtm(corpus)
print("DTM shape (documents x terms):", dtm.shape)
dtm

In [ ]:
total_cells = dtm.size
zero_cells = (dtm == 0).sum().sum()
print(f"Total cells:      {total_cells}")
print(f"Zero cells:       {zero_cells}")
print(f"Percentage zeros: {100 * zero_cells / total_cells:.1f}%")

In [ ]:
np.random.seed(0)
sizes = [2, 4, 6, 8]
vocab_sizes = []
sparsities = []
for n in sizes:
    d = make_dtm(corpus[:n])
    vocab_sizes.append(d.shape[1])
    sparsities.append(100 * (d == 0).sum().sum() / d.size)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(sizes, vocab_sizes, marker="o", color="#34B233")
ax1.set_title("Vocabulary grows with corpus size")
ax1.set_xlabel("Number of documents"); ax1.set_ylabel("Vocabulary size (columns)")
ax1.grid(alpha=0.3)
ax2.plot(sizes, sparsities, marker="o", color="#1A1A2E")
ax2.set_title("And the matrix gets emptier")
ax2.set_xlabel("Number of documents"); ax2.set_ylabel("Percentage of zero cells")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **✏️ Exercise 3**
>
> Add two or three of your own sentences on new topics with new words. Does the vocabulary
> get wider and the sparsity higher?


In [ ]:
# ✅ Solution
corpus_extended = corpus + [
    "astronauts repaired the space station",
    "the chef prepared a delicious dessert",
    "hikers climbed the steep mountain trail",
]

dtm_small = make_dtm(corpus)
dtm_big = make_dtm(corpus_extended)

def sparsity(d):
    return 100 * (d == 0).sum().sum() / d.size

print(f"Original:  {dtm_small.shape[1]} words, {sparsity(dtm_small):.1f}% zeros")
print(f"Extended:  {dtm_big.shape[1]} words, {sparsity(dtm_big):.1f}% zeros")

# Comment: yes — each new sentence on a fresh topic introduces mostly new words, so
# the vocabulary (columns) grows while each document still fills only a few cells.
# The percentage of zeros rises. Scale this to thousands of documents and you get the
# familiar >99%-sparse matrices of real corpora.

## 4. Out-of-vocabulary words

In [ ]:
train_docs = ["the policy is good"]
vec = CountVectorizer()
vec.fit(train_docs)
print("Vocabulary the model knows:", list(vec.get_feature_names_out()))

In [ ]:
new_doc = ["the policy is catastrophic"]
X_new = vec.transform(new_doc)
result = pd.DataFrame(X_new.toarray(), columns=vec.get_feature_names_out(), index=["new_doc"])
print("New sentence:", new_doc[0])
print()
print("How the model represents it:")
print(result)

In [ ]:
bland = vec.transform(["the policy is"])
catastrophic = vec.transform(["the policy is catastrophic"])
print("Are the two vectors identical?", np.array_equal(bland.toarray(), catastrophic.toarray()))
print("\n'catastrophic' contributed nothing — it was out of vocabulary.")

> **✏️ Exercise 4**
>
> Transform `"the wonderful brilliant policy"` using the same `vec`. How many of its words
> survive? What does that tell you about applying a fitted model to new text?


In [ ]:
# ✅ Solution
test = ["the wonderful brilliant policy"]
X_test = vec.transform(test)
result = pd.DataFrame(X_test.toarray(), columns=vec.get_feature_names_out(), index=["test"])
print(result)
print()
survived = X_test.sum()
print(f"Words in the sentence: 4  ('the', 'wonderful', 'brilliant', 'policy')")
print(f"Words that survived:   {survived}  (only the ones in the training vocab)")

# Comment: only "the" and "policy" survive — "wonderful" and "brilliant" were never
# seen at fit time, so they are dropped. On real text this is a serious problem: any
# model you fit on one corpus will be blind to the many words that appear only in new
# data (new slang, names, typos, domain terms). The richer the new text, the more you
# silently lose.

## 5. One meaning per word (context is ignored)

In [ ]:
sentences = [
    "I sat on the river bank",
    "I fished from the river bank",
    "I deposited cash at the bank",
]
make_dtm(sentences)

In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(sentences)
sims = cosine_similarity(X)
sim_df = pd.DataFrame(
    sims.round(3),
    index=["D1 river", "D2 river", "D3 money"],
    columns=["D1 river", "D2 river", "D3 money"],
)
print("Similarity — note it is driven purely by shared word strings:")
sim_df

> **✏️ Exercise 5**
>
> Write three sentences using **"spring"** in different senses (season, coil, to jump).
> Build the DTM. Confirm there is only one "spring" column, and explain why that is a problem.


In [ ]:
# ✅ Solution
spring_sentences = [
    "the flowers bloom in spring",       # season
    "the mattress has a broken spring",  # coil
    "the cat will spring onto the table", # to jump
]
dtm = make_dtm(spring_sentences)
print(dtm)
print()
print("Number of columns named 'spring':", list(dtm.columns).count("spring"))

# Comment: there is exactly ONE "spring" column, and all three sentences put a 1 in it,
# even though the word means three completely different things. Bag-of-words has no way
# to represent these distinct senses — context (the surrounding words that tell a human
# which meaning is intended) is discarded. Any analysis that keys on "spring" will
# silently mix seasons, coils, and jumping.

## Wrap-up

That's the full solution set. Each exercise reinforces one structural limitation of
bag-of-words — none of which can be fixed by cleaning harder or adding more words.

### Optional challenge

Rescue word order with **bigrams** and see what it costs.


In [ ]:
# ✅ Solution
pair = [
    "the dog bit the man",
    "the man bit the dog",
]

# Unigrams only (baseline)
X_uni = CountVectorizer(ngram_range=(1, 1)).fit_transform(pair)
sim_uni = cosine_similarity(X_uni)[0, 1]

# Unigrams + bigrams
vec_bi = CountVectorizer(ngram_range=(1, 2))
X_bi = vec_bi.fit_transform(pair)
sim_bi = cosine_similarity(X_bi)[0, 1]

print(f"Similarity with unigrams only:      {sim_uni:.3f}")
print(f"Similarity with unigrams + bigrams: {sim_bi:.3f}")
print()
print(f"Columns with unigrams only:      {CountVectorizer(ngram_range=(1,1)).fit(pair).transform(pair).shape[1]}")
print(f"Columns with unigrams + bigrams: {X_bi.shape[1]}")
print()
print("Bigram features that capture order:")
print([f for f in vec_bi.get_feature_names_out() if " " in f])

# Comment: bigrams DO drop the similarity below 1.0, because "dog bit" and "bit the
# man" now differ from "man bit" and "bit the dog". So order is partially rescued.
# BUT the cost is a much wider matrix — we roughly doubled the number of columns for
# just two short sentences. On real corpora, adding bigrams (let alone trigrams)
# explodes the vocabulary and makes the sparsity problem (drawback 3) far worse.
# It is a patch, not a cure.